# Regime Diagnostic: Thermodynamic Profiles and Organism Outcomes in Evochora

This notebook analyzes simulation data from the digital evolution platform [Evochora](https://github.com/evochora/evochora) (v0.4.0) to investigate the following question:

**Do organisms with similar thermodynamic profiles (energy and entropy levels) consistently show similar outcomes (persistence, reproductive success), or are there cases where similar thermodynamic characteristics lead to different behaviors?**

This question is motivated by the hierarchical life framework proposed by Zeid (2026, unpublished manuscript: *"Life: A General Framework Across Substrates"*), which defines a hierarchy of dynamical regimes:

- **Physical Life**: Systems that maintain organized structure through sustained energy flux, without encoded heritable information
- **Informational Life**: Systems that additionally replicate encoded patterns with variation, undergoing selection

If thermodynamic variables alone explain organism outcomes, the system operates in what the framework calls the Physical Life regime. If organisms with similar thermodynamic profiles show systematically different outcomes depending on their genome, this would suggest a role for informational structure beyond thermodynamics.

We operationalize "outcomes" along two measurable axes (following Zeid's definition of adaptation as a multi-dimensional profile):

| Axis | Operationalization |
|------|-------------------|
| **Persistence** | Lifespan (ticks from birth to death) |
| **Reproductive success** | Number of offspring produced |

For each axis, organisms are grouped into quintiles by both **energy** and **entropy** separately, and within-group variance is analyzed to determine whether genome identity explains outcome differences beyond what thermodynamic profile predicts.

A third axis proposed by Zeid — **informational influence** (the extent to which an organism's internal state influences future system dynamics) — is not computed in this notebook. The challenges of operationalizing this measure in Evochora are discussed in the [Limitations](#limitations) section.

## 2. System Overview

[Evochora](https://github.com/evochora/evochora) is an open-source simulation platform for digital evolution research where organisms are self-replicating programs in a configurable spatial environment. For a comprehensive description, see the [Scientific Overview](https://github.com/evochora/evochora/blob/v0.4.0/docs/SCIENTIFIC_OVERVIEW.md).

The aspects relevant to this analysis:

- **Organisms** are independent virtual machines executing one instruction per tick in a spatial grid. They interact with the environment exclusively through local pointers and have no global view of the simulation (see [Section 2.2 of the Scientific Overview](https://github.com/evochora/evochora/blob/v0.4.0/docs/SCIENTIFIC_OVERVIEW.md#22-the-virtual-machine-and-spatial-code-execution)).

- **Survival** is governed by two state variables: an energy register (ER) and an entropy register (SR). Each instruction costs energy and produces entropy; both costs are configurable per instruction type. An organism dies when ER ≤ 0 or when SR ≥ the configured maximum. Organisms replenish energy by absorbing ENERGY molecules from the environment and reduce entropy by writing molecules via POKE instructions. Details in [Section 2.3 of the Scientific Overview](https://github.com/evochora/evochora/blob/v0.4.0/docs/SCIENTIFIC_OVERVIEW.md#23-metabolism-survival-and-ownership).

- **Reproduction** occurs when an organism copies its code into adjacent space and executes FORK, transferring ownership of the copied cells to a new child organism.

- **Mutation** is applied at birth through four configurable operators: substitution, insertion, deletion, and duplication (see [Section 2.4 of the Scientific Overview](https://github.com/evochora/evochora/blob/v0.4.0/docs/SCIENTIFIC_OVERVIEW.md#24-extensible-by-design-the-plugin-architecture)).

- **Genome identity** is a 64-bit hash computed over all non-DATA molecules owned by an organism. A hash of 0 means the organism has no code of its own.

## 3. Simulation Configuration

The analyzed simulation was produced with Evochora [v0.4.0](https://github.com/evochora/evochora/releases/tag/v0.4.0). The configuration is specific to the analyzed run and loaded dynamically below.

The parameters most relevant to this analysis fall into three categories:

- **Thermodynamics**: Energy and entropy costs per instruction, maximum energy and entropy, error penalty. These define the selection pressure organisms face. See the [thermodynamic configuration](https://github.com/evochora/evochora/blob/v0.4.0/config/evochora.conf#L55-L119) in the default config for reference.

- **Mutation**: Rates and mechanisms for substitution, insertion, deletion, and duplication applied at birth. See the [mutation configuration](https://github.com/evochora/evochora/blob/v0.4.0/config/evochora.conf#L127-L174) in the default config.

- **Energy supply**: How and at what rate ENERGY molecules are spawned in the environment.

The code cell below loads and displays the configuration of the analyzed run, highlighting deviations from the [v0.4.0 default configuration](https://github.com/evochora/evochora/blob/v0.4.0/config/evochora.conf).

In [20]:
# --- Setup & Run Selection ---
import subprocess, sys, json, warnings
warnings.filterwarnings("ignore")

try:
    import requests, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
    from scipy import stats
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "requests", "pandas", "numpy", "matplotlib", "seaborn", "scipy"])
    import requests, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
    from scipy import stats

# --- Configuration ---
BASE_URL = "https://evochora.org"       # Public demo
#BASE_URL = "http://localhost:8081"      # Local instance

# Uncomment to analyze a specific run (default: most recent)
#RUN_ID = "20260227-14120409-e3932a5a-723a-49f7-b45e-23b22021ceed"  # 265M ticks, more pressure
RUN_ID = "20260226-03114337-11f7e3fc-d67c-4012-930d-cea20143b3b5"  # 2B ticks, low pressure

def api_get(url, **kwargs):
    resp = requests.get(url, **kwargs)
    resp.raise_for_status()
    return resp.json()

# Select run
runs = api_get(f"{BASE_URL}/analyzer/api/runs")
if "RUN_ID" not in dir():
    RUN_ID = max(runs, key=lambda r: r.get("startTime", 0))["runId"]
run_params = {"runId": RUN_ID}

# Load metadata
ticks_meta = api_get(f"{BASE_URL}/visualizer/api/organisms/ticks", params=run_params)
MIN_TICK, MAX_TICK = ticks_meta["minTick"], ticks_meta["maxTick"]
sim_metadata = api_get(f"{BASE_URL}/visualizer/api/simulation/metadata", params=run_params)
config = json.loads(sim_metadata.get("resolvedConfigJson", "{}"))

print(f"Run:        {RUN_ID}")
print(f"Tick range: {MIN_TICK:,} – {MAX_TICK:,}")

Run:        20260226-03114337-11f7e3fc-d67c-4012-930d-cea20143b3b5
Tick range: 0 – 2,017,450,000


In [21]:
# --- Display scientifically relevant configuration ---
runtime_cfg = config.get("runtime", {})
organism_cfg = runtime_cfg.get("organism", {})
thermo_cfg = runtime_cfg.get("thermodynamics", {})
env_cfg = config.get("environment", {})
organisms_cfg = config.get("organisms", [{}])

# v0.4.0 defaults for deviation comparison
DEFAULTS = {
    "SeedEnergyCreator": {"percentage": 0.0025, "amount": 10000, "amountVariance": 0.2},
    "GeyserCreator": {"percentage": 0.0002, "interval": 100, "amount": 10000, "safetyRadius": 3},
    "SolarRadiationCreator": {"probability": 0.02, "amount": 10000, "safetyRadius": 1, "executionsPerTick": 1},
    "GeneDuplicationPlugin": {"duplicationRate": 0.1, "minNopSize": 8},
    "GeneDeletionPlugin": {"deletionRate": 0.025, "countExponent": 2},
    "GeneInsertionPlugin": {"mutationRate": 0.05},
    "GeneSubstitutionPlugin": {"substitutionRate": 0.025},
}

print("=" * 65)
print("SIMULATION CONFIGURATION")
print("=" * 65)

print(f"\n{'ENVIRONMENT'}")
print(f"  Grid shape:    {env_cfg.get('shape', '?')}")
print(f"  Topology:      {env_cfg.get('topology', '?')}")

print(f"\n{'INITIAL ORGANISMS'}")
for i, org in enumerate(organisms_cfg):
    print(f"  Organism {i+1}:")
    print(f"    Program:        {org.get('program', '?')}")
    print(f"    Initial energy: {org.get('initialEnergy', '?'):,}")
    print(f"    Placement:      {org.get('placement', {}).get('positions', '?')}")

print(f"\n{'THERMODYNAMICS'}")
print(f"  Max energy:        {organism_cfg.get('max-energy', '?'):,}")
print(f"  Max entropy:       {organism_cfg.get('max-entropy', '?'):,}")
print(f"  Error penalty:     {organism_cfg.get('error-penalty-cost', '?')} energy per failed instruction")
print(f"  Max skips/tick:    {organism_cfg.get('max-skips-per-tick', '?')}")
default_thermo = thermo_cfg.get("default", {}).get("options", {})
print(f"  Base energy cost:  {default_thermo.get('base-energy', '?')} per instruction")
print(f"  Base entropy cost: {default_thermo.get('base-entropy', '?')} per instruction")

# POKE entropy reduction from write-rules
overrides = thermo_cfg.get("overrides", {}).get("instructions", {})
for key, override in overrides.items():
    if "POKE" in key:
        write_rules = override.get("options", {}).get("write-rules", {})
        default_write = write_rules.get("_default", {})
        print(f"  POKE entropy:      {default_write.get('entropy', '?')} per write")
        break

print(f"\n{'SEED'}")
print(f"  Random seed:       {config.get('seed', '?')}")

print(f"\n{'PLUGINS'}")
plugins = config.get("plugins", [])
deviations = []
for plugin in plugins:
    class_name = plugin.get("className", "").split(".")[-1]
    options = plugin.get("options", {})
    
    # Skip non-scientific plugins
    if class_name in ("DecayOnDeath", "LabelRewritePlugin"):
        continue
    
    print(f"\n  {class_name}:")
    defaults = DEFAULTS.get(class_name, {})
    for key, value in sorted(options.items()):
        if isinstance(value, (dict, list)):
            continue  # skip nested config (entries, CODE, etc.)
        default = defaults.get(key)
        if default is not None and value != default:
            print(f"    {key}: {value}  ⚠ (default: {default})")
            deviations.append((class_name, key, value, default))
        else:
            print(f"    {key}: {value}")

if deviations:
    print(f"\n{'DEVIATIONS FROM v0.4.0 DEFAULTS':}")
    for cls, key, val, default in deviations:
        print(f"  {cls}.{key}: {val} (default: {default})")
else:
    print(f"\n  No deviations from v0.4.0 defaults.")

SIMULATION CONFIGURATION

ENVIRONMENT
  Grid shape:    [4096, 2304]
  Topology:      TORUS

INITIAL ORGANISMS
  Organism 1:
    Program:        assembly/primordial/main.evo
    Initial energy: 100,000
    Placement:      [175, 135]

THERMODYNAMICS
  Max energy:        150,000
  Max entropy:       10,000
  Error penalty:     100 energy per failed instruction
  Max skips/tick:    100
  Base energy cost:  1 per instruction
  Base entropy cost: 1 per instruction
  POKE entropy:      -500 per write

SEED
  Random seed:       42

PLUGINS

  SeedEnergyCreator:
    amount: 10000
    amountVariance: 0.2
    percentage: 0.0025

  GeyserCreator:
    amount: 10000
    interval: 100
    percentage: 0.0002
    safetyRadius: 3

  SolarRadiationCreator:
    amount: 10000
    executionsPerTick: 1
    probability: 0.02
    safetyRadius: 1

  GeneDuplicationPlugin:
    duplicationRate: 0.1
    minNopSize: 8

  GeneDeletionPlugin:
    countExponent: 2
    deletionRate: 0.025

  GeneInsertionPlugin:
    mu

## 4. Methodology

### 4.1 Data Sources

1. **Analyzer metrics** (Parquet via Analyzer API): Population-level time series — population size, birth counts, genome diversity, and generation depth.

2. **Per-organism snapshots** (Visualizer API): At each sampled tick, all organisms with their energy (ER), entropy (SR), genome hash, parent ID, birth tick, and death tick. Dead organisms appear exactly once — in the first snapshot after their death.

### 4.2 Per-Organism Lifecycle Construction

From the snapshots, we construct a lifecycle record for each organism: birth/death ticks, lifespan, average energy and entropy over lifetime, energy and entropy at death, offspring count (via parent ID), and genome hash.

### 4.3 Data Cleaning

Three categories of organisms are excluded from the main analysis:

1. **Genome-0 organisms** (genome hash = 0) have no code of their own. They arise when a parent's reproduction mechanism is faulty and FORK executes without transferring molecules to the child.

2. **Single-snapshot organisms** appeared alive in fewer than 2 sampled ticks. For these organisms, the only recorded energy and entropy values are from their death snapshot, which reflects their state at death rather than their lifetime thermodynamic profile. Including them would conflate death-state measurements with lifetime averages.

Both groups are reported in the data summary but excluded from the within-quintile analysis.

**Death cause** is classified as energy depletion (ER ≤ 0) or entropy overflow (SR ≥ maximum).

### 4.4 Thermodynamic Grouping

Analyzed organisms are divided into quintiles separately by average energy and by average entropy. Within each quintile, a one-way ANOVA tests whether genome identity explains a significant portion of the outcome variance. The effect size η² (eta-squared) measures the proportion of variance explained by genome.

### 4.5 Scope

All results are descriptive observations on a single simulation run. Statistical limitations based on the observed population sizes are discussed in the [Limitations](#limitations) section.

## 5. Data Loading & Preparation

In [22]:
# --- Fetch analyzer metrics (Parquet) ---
import io
from concurrent.futures import ThreadPoolExecutor

METRICS = ["population", "vital_stats", "genome_diversity", "generation_depth"]

def _fetch_parquet(metric):
    for lod in ["lod0", "lod1", "lod2"]:
        resp = requests.get(f"{BASE_URL}/analyzer/api/parquet",
                            params={**run_params, "metric": metric, "lod": lod})
        if resp.status_code == 200 and len(resp.content) > 100:
            return metric, pd.read_parquet(io.BytesIO(resp.content)), lod
    return metric, pd.DataFrame(), "none"

print("Fetching analyzer metrics...", flush=True)
with ThreadPoolExecutor(max_workers=len(METRICS)) as pool:
    results = list(pool.map(_fetch_parquet, METRICS))

metrics = {}
for metric, df, lod in results:
    metrics[metric] = df
    print(f"  {metric}: {len(df):,} rows (LOD: {lod})")

pop = metrics["population"]
vital = metrics["vital_stats"]
genome_div = metrics["genome_diversity"]
gen_depth = metrics["generation_depth"]

Fetching analyzer metrics...
  population: 40,350 rows (LOD: lod0)
  vital_stats: 40,350 rows (LOD: lod0)
  genome_diversity: 40,350 rows (LOD: lod0)
  generation_depth: 40,350 rows (LOD: lod0)


In [ ]:
# --- Fetch per-organism snapshots ---
import time

sampling_interval = config.get("samplingInterval", 50000)
all_ticks = list(range(MIN_TICK, MAX_TICK + 1, sampling_interval))
print(f"Fetching {len(all_ticks):,} organism snapshots ({sampling_interval:,}-tick intervals)...", flush=True)

all_snapshots = {}

def _fetch_tick(tick):
    for attempt in range(3):
        try:
            resp = requests.get(f"{BASE_URL}/visualizer/api/organisms/{tick}",
                                params=run_params, timeout=60)
            resp.raise_for_status()
            return tick, resp.json().get("organisms", [])
        except Exception as e:
            if attempt == 2:
                print(f"  Failed tick {tick}: {e}", flush=True)
                return tick, []
            time.sleep(2)

start = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    done = 0
    for tick, orgs in pool.map(_fetch_tick, all_ticks):
        all_snapshots[tick] = orgs
        done += 1
        if done % 1000 == 0:
            elapsed = time.time() - start
            print(f"  {done:,}/{len(all_ticks):,} ({elapsed:.0f}s)", flush=True)

elapsed = time.time() - start
total_records = sum(len(orgs) for orgs in all_snapshots.values())
print(f"Done in {elapsed:.0f}s — {len(all_snapshots):,} snapshots, {total_records:,} organism-tick records")

Fetching 40,350 organism snapshots (50,000-tick intervals)...


In [ ]:
# --- Build per-organism lifecycle table ---
from collections import defaultdict

MAX_ENTROPY = config.get("runtime", {}).get("organism", {}).get("max-entropy", 10000)

organism_data = {}
offspring_count = defaultdict(int)

for tick in sorted(all_snapshots.keys()):
    for o in all_snapshots[tick]:
        oid = o["organismId"]
        if oid not in organism_data:
            organism_data[oid] = {
                "organismId": oid,
                "parentId": o.get("parentId"),
                "birthTick": o["birthTick"],
                "genomeHash": str(o["genomeHash"]),
                "deathTick": -1,
                "isDead": False,
                "energy_samples": [],
                "entropy_samples": [],
                "aliveSamples": 0,
            }
        od = organism_data[oid]
        if not od["isDead"]:
            od["energy_samples"].append(o["energy"])
            od["entropy_samples"].append(o["entropyRegister"])
            od["aliveSamples"] += 1
        if o.get("isDead", False):
            od["isDead"] = True
            od["deathTick"] = o.get("deathTick", tick)
            od["energyAtDeath"] = o["energy"]
            od["entropyAtDeath"] = o["entropyRegister"]

for od in organism_data.values():
    if od["parentId"] is not None:
        offspring_count[od["parentId"]] += 1

# --- Build DataFrame ---
records = []
for oid, od in organism_data.items():
    birth = od["birthTick"]
    death = od["deathTick"]
    lifespan = (death - birth) if death > 0 else (MAX_TICK - birth)
    energies = od["energy_samples"]
    entropies = od["entropy_samples"]

    records.append({
        "organismId": oid,
        "parentId": od["parentId"],
        "genomeHash": od["genomeHash"],
        "birthTick": birth,
        "deathTick": death,
        "lifespan": lifespan,
        "isAlive": not od["isDead"],
        "avgEnergy": np.mean(energies) if energies else np.nan,
        "avgEntropy": np.mean(entropies) if entropies else np.nan,
        "energyAtDeath": od.get("energyAtDeath", np.nan),
        "entropyAtDeath": od.get("entropyAtDeath", np.nan),
        "offspring": offspring_count.get(oid, 0),
        "aliveSamples": od["aliveSamples"],
        "isGenome0": od["genomeHash"] == "0",
    })

df = pd.DataFrame(records)

# --- Classify populations ---
genome0 = df[df["isGenome0"]]
viable_all = df[~df["isGenome0"]]
single_snapshot = viable_all[viable_all["aliveSamples"] < 2]
analyzed = viable_all[viable_all["aliveSamples"] >= 2].copy()

# Death cause for viable dead
viable_dead = viable_all[~viable_all["isAlive"]].copy()
viable_dead["deathCause"] = np.where(
    viable_dead["energyAtDeath"] <= 0, "energy_depletion",
    np.where(viable_dead["entropyAtDeath"] >= MAX_ENTROPY, "entropy_overflow", "unknown")
)

print(f"Total organisms:             {len(df):,}")
print(f"  Genome-0 (excluded):       {len(genome0):,} ({len(genome0)/len(df)*100:.1f}%)")
print(f"  Viable:                    {len(viable_all):,}")
print(f"    Single-snapshot (excl.): {len(single_snapshot):,} ({len(single_snapshot)/len(viable_all)*100:.1f}% of viable)")
print(f"    Analyzed (≥2 samples):   {len(analyzed):,} ({len(analyzed)/len(viable_all)*100:.1f}% of viable)")
print(f"  Unique genomes (analyzed): {analyzed['genomeHash'].nunique():,}")

## 6. Results

### 6.1 Population Context

The replication index Π = ⟨r⟩ / γ_decay ([Zeid, Section 3.5.2](https://github.com/evochora/evochora)) measures whether replication outpaces decay. Π > 1 indicates informational persistence across generations; Π ≈ 1 indicates dynamic equilibrium.

In [ ]:
# --- 6.1 Population context & Replication Index Π ---
plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
    "axes.titlesize": 12, "axes.labelsize": 10, "legend.fontsize": 9,
    "font.family": "sans-serif",
})

fig, ax1 = plt.subplots(figsize=(14, 4))

ax1.plot(pop["tick"], pop["alive_count"], color="#2ca02c", lw=0.8, alpha=0.8)
ax1.set_xlabel("Tick")
ax1.set_ylabel("Alive organisms", color="#2ca02c")
ax1.tick_params(axis="y", labelcolor="#2ca02c")
ax1.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

# Replication Index Π
vital_c = vital.copy()
vital_c["births"] = vital_c["total_born"].diff().fillna(0)
vital_c["deaths"] = vital_c["births"] - vital_c["alive_count"].diff().fillna(0)
tick_step = vital_c["tick"].diff().median()
window_ticks = 10_000_000
window_rows = max(1, int(window_ticks / tick_step)) if tick_step and tick_step > 0 else 1
vital_c["births_roll"] = vital_c["births"].rolling(window_rows, min_periods=1).sum()
vital_c["deaths_roll"] = vital_c["deaths"].rolling(window_rows, min_periods=1).sum()
vital_c["pi"] = vital_c["births_roll"] / vital_c["deaths_roll"].replace(0, np.nan)
valid_pi = vital_c.dropna(subset=["pi"])
valid_pi = valid_pi[(valid_pi["pi"] > 0) & (valid_pi["pi"] < 1000)]

ax2 = ax1.twinx()
ax2.plot(valid_pi["tick"], valid_pi["pi"], color="#d62728", lw=0.6, alpha=0.5)
ax2.axhline(1.0, color="#d62728", ls="--", lw=1, alpha=0.4)
ax2.set_ylabel("Replication Index Π", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")

fig.suptitle("Population size and Replication Index Π over time", fontsize=12)
plt.tight_layout()
plt.show()

stable_pi = valid_pi[valid_pi["tick"] > MAX_TICK * 0.1]
print(f"Population — min: {pop['alive_count'].min()}, max: {pop['alive_count'].max()}, "
      f"mean: {pop['alive_count'].mean():.0f}")
print(f"Π (after initial transient) — mean: {stable_pi['pi'].mean():.4f}, std: {stable_pi['pi'].std():.4f}")
print(f"Total organisms ever born: {vital['total_born'].iloc[-1]:,}")

### 6.2 Data Summary

In [ ]:
# --- 6.2 Data summary ---
from IPython.display import display, Markdown

total_dead_viable = int((~viable_all["isAlive"]).sum())
summary_rows = [
    ("Total organisms", len(df), ""),
    ("Genome-0 (no code)", len(genome0), f"{len(genome0)/len(df)*100:.1f}%"),
    ("Viable organisms", len(viable_all), f"{len(viable_all)/len(df)*100:.1f}%"),
    ("  — single-snapshot (no lifetime data)", len(single_snapshot), f"{len(single_snapshot)/len(viable_all)*100:.1f}% of viable"),
    ("  — **analyzed (≥2 alive samples)**", len(analyzed), f"{len(analyzed)/len(viable_all)*100:.1f}% of viable"),
]
if total_dead_viable > 0:
    for cause in ["energy_depletion", "entropy_overflow", "unknown"]:
        n = int((viable_dead["deathCause"] == cause).sum())
        if n > 0:
            summary_rows.append((f"Death cause: {cause.replace('_', ' ')}", n, f"{n/total_dead_viable*100:.1f}% of viable dead"))
summary_rows.append(("Unique genomes (analyzed)", int(analyzed["genomeHash"].nunique()), ""))

md = "| Metric | Count | Share |\n|--------|------:|-------|\n"
for label, count, share in summary_rows:
    md += f"| {label} | {count:,} | {share} |\n"
display(Markdown(md))

### 6.3 Central Question: Do Genomes Matter Beyond Thermodynamics?

For each outcome axis, organisms are divided into quintiles by energy and by entropy. Within each quintile, a one-way ANOVA tests whether lineage grouping explains a significant portion of the outcome variance. The analysis is repeated at three lineage depths (1 = individual genomes, 3 and 5 = progressively coarser lineage families) to assess sensitivity to grouping granularity.

Additionally, rolling averages over simulation time show whether outcomes improve as the population evolves.

<a id="persistence-def"></a><a id="reproduction-def"></a>

#### 6.3.1 Persistence (Lifespan)

Lifespan: ticks from birth to death.

In [ ]:
# --- Fetch lineage tree & build ancestor lookup ---
lineage_resp = api_get(f"{BASE_URL}/visualizer/api/organisms/{MAX_TICK}", params=run_params)
lineage_tree = lineage_resp.get("genomeLineageTree", {})
print(f"Genome lineage tree: {len(lineage_tree):,} genomes")

def find_ancestor_at_depth(genome, tree, max_depth):
    """Walk up max_depth steps in the genome lineage tree."""
    current = genome
    for _ in range(max_depth):
        parent = tree.get(current)
        if parent is None:
            break
        current = parent
    return current

# --- ANOVA analysis with lineage depth comparison ---
from matplotlib.patches import Patch

DEPTHS = [1, 3, 5]
DEPTH_COLORS = ["#e41a1c", "#377eb8", "#4daf4a"]

def anova_lineage_analysis(data, grouping_col, outcome_col, grouping_label, outcome_label, log_y=False):
    """
    Combined chart: boxplots + stripplots for 3 lineage depths side by side per quintile.
    Plus ANOVA summary table for all depths.
    """
    df_a = data.copy()
    df_a["quintile"] = pd.qcut(df_a[grouping_col], q=5, labels=False) + 1
    q_bounds = df_a.groupby("quintile")[grouping_col].agg(["min", "max"])
    quintiles = sorted(df_a["quintile"].unique())
    n_per_q = len(df_a) // 5
    
    fig, ax = plt.subplots(figsize=(16, 6))
    width = 0.25
    
    anova_all = []
    
    for d_idx, depth in enumerate(DEPTHS):
        if depth == 1:
            df_a["group"] = df_a["genomeHash"]
        else:
            df_a["group"] = df_a["genomeHash"].apply(
                lambda g: find_ancestor_at_depth(g, lineage_tree, depth))
        
        n_groups = df_a["group"].nunique()
        
        for i, q in enumerate(quintiles):
            grp = df_a[df_a["quintile"] == q]
            gm = grp.groupby("group")[outcome_col].agg(["mean", "count"])
            
            pos = i + (d_idx - 1) * width
            
            # Boxplot
            bp = ax.boxplot([gm["mean"].values], positions=[pos], widths=width * 0.8,
                           patch_artist=True, showfliers=False,
                           medianprops=dict(color="black", lw=1.5),
                           whiskerprops=dict(alpha=0.4), capprops=dict(alpha=0.4))
            bp["boxes"][0].set_facecolor(DEPTH_COLORS[d_idx])
            bp["boxes"][0].set_alpha(0.4)
            
            # Stripplot (genome/lineage means as dots)
            jitter = np.random.default_rng(42 + q * 10 + d_idx).uniform(-width * 0.3, width * 0.3, len(gm))
            sizes = np.clip(gm["count"].values * 1.5, 5, 50)
            ax.scatter(pos + jitter, gm["mean"].values, s=sizes,
                      alpha=0.5, color=DEPTH_COLORS[d_idx], edgecolors="white",
                      linewidths=0.3, zorder=5)
        
        # ANOVA per quintile
        for q in quintiles:
            grp = df_a[df_a["quintile"] == q]
            groups = [g[outcome_col].values for _, g in grp.groupby("group") if len(g) >= 2]
            if len(groups) >= 2:
                f_val, p_val = stats.f_oneway(*groups)
                ss_b = sum(len(g) * (g.mean() - grp[outcome_col].mean())**2
                          for g in [grp[grp["group"] == gh][outcome_col]
                                    for gh in grp["group"].unique()])
                ss_t = ((grp[outcome_col] - grp[outcome_col].mean())**2).sum()
                eta = ss_b / ss_t if ss_t > 0 else 0
                anova_all.append((depth, q, f_val, p_val, eta, len(groups), n_groups))
            else:
                anova_all.append((depth, q, np.nan, np.nan, np.nan, 0, n_groups))
    
    # Labels
    q_labels = [f"Q{q}\n{q_bounds.loc[q, 'min']:,.0f}–\n{q_bounds.loc[q, 'max']:,.0f}" for q in quintiles]
    ax.set_xticks(range(len(quintiles)))
    ax.set_xticklabels(q_labels, fontsize=9)
    ax.set_xlabel(f"{grouping_label} quintile")
    ax.set_ylabel(f"Mean {outcome_label.lower()} per lineage group")
    if log_y:
        ax.set_yscale("log")
    
    ax.set_title(f"{outcome_label} by {grouping_label} quintile — lineage depth comparison\n"
                 f"n≈{n_per_q:,} organisms/quintile · dots = lineage group means (size ∝ group size)")
    
    legend_elements = [Patch(facecolor=c, alpha=0.5, label=f"Depth {d}")
                       for d, c in zip(DEPTHS, DEPTH_COLORS)]
    ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # ANOVA summary table
    print(f"\nANOVA summary: does lineage grouping explain {outcome_label.lower()} variance "
          f"within {grouping_label.lower()} quintiles?\n")
    print(f"{'Depth':>6}  {'Q':>3}  {'F':>8}  {'p-value':>12}  {'η²':>6}  {'Groups(≥2)':>11}  {'Sig':>5}")
    for depth, q, f_val, p_val, eta, n_g, total_g in anova_all:
        if np.isnan(f_val):
            print(f"  {depth:>4}  {q:>3}  {'—':>8}  {'—':>12}  {'—':>6}  {n_g:>11}  {'—':>5}")
        else:
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  {depth:>4}  {q:>3}  {f_val:>8.2f}  {p_val:>12.2e}  {eta:>6.3f}  {n_g:>11}  {sig:>5}")
    print(f"\n  η² = proportion of variance explained by lineage group (0 = none, 1 = all)")
    print(f"  Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

In [ ]:
# --- 6.3a Persistence by Energy quintile ---
anova_lineage_analysis(analyzed, "avgEnergy", "lifespan", "Energy", "Lifespan (ticks)", log_y=True)

In [ ]:
# --- 6.3b Persistence by Entropy quintile ---
anova_lineage_analysis(analyzed, "avgEntropy", "lifespan", "Entropy", "Lifespan (ticks)", log_y=True)

In [ ]:
# --- Lifespan over simulation time ---
dead_analyzed = analyzed[~analyzed["isAlive"]].copy().sort_values("birthTick")
window = max(1, len(dead_analyzed) // 20)
dead_analyzed["lifespan_roll"] = dead_analyzed["lifespan"].rolling(window, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.scatter(dead_analyzed["birthTick"], dead_analyzed["lifespan"], s=1, alpha=0.1, color="grey")
ax.plot(dead_analyzed["birthTick"], dead_analyzed["lifespan_roll"], color="red", lw=2)
ax.set_yscale("log")
ax.set_xlabel("Birth tick")
ax.set_ylabel("Lifespan (ticks, log scale)")
ax.set_title("Lifespan over simulation time (dead organisms only, rolling average in red)")
ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
plt.tight_layout()
plt.show()

early = dead_analyzed[dead_analyzed["birthTick"] < dead_analyzed["birthTick"].quantile(0.25)]
late = dead_analyzed[dead_analyzed["birthTick"] > dead_analyzed["birthTick"].quantile(0.75)]
print(f"Early 25%: median lifespan = {early['lifespan'].median():,.0f}")
print(f"Late 25%:  median lifespan = {late['lifespan'].median():,.0f}")
print(f"Note: organisms still alive at end of simulation are not included.")

#### 6.3.2 Reproductive Success (Offspring)

Offspring: number of children produced by an organism.

In [ ]:
# --- 6.3c Reproductive success by Energy quintile ---
anova_lineage_analysis(analyzed, "avgEnergy", "offspring", "Energy", "Offspring count")

In [ ]:
# --- 6.3d Reproductive success by Entropy quintile ---
anova_lineage_analysis(analyzed, "avgEntropy", "offspring", "Entropy", "Offspring count")

In [ ]:
# --- Offspring over simulation time ---
dead_analyzed_off = analyzed[~analyzed["isAlive"]].copy().sort_values("birthTick")
window = max(1, len(dead_analyzed_off) // 20)
dead_analyzed_off["offspring_roll"] = dead_analyzed_off["offspring"].rolling(window, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.scatter(dead_analyzed_off["birthTick"], dead_analyzed_off["offspring"], s=1, alpha=0.15, color="grey")
ax.plot(dead_analyzed_off["birthTick"], dead_analyzed_off["offspring_roll"], color="red", lw=2)
ax.set_xlabel("Birth tick")
ax.set_ylabel("Offspring count")
ax.set_title("Offspring over simulation time (dead organisms only, rolling average in red)")
ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
plt.tight_layout()
plt.show()

early = dead_analyzed_off[dead_analyzed_off["birthTick"] < dead_analyzed_off["birthTick"].quantile(0.25)]
late = dead_analyzed_off[dead_analyzed_off["birthTick"] > dead_analyzed_off["birthTick"].quantile(0.75)]
print(f"Early 25%: mean offspring = {early['offspring'].mean():.2f}")
print(f"Late 25%:  mean offspring = {late['offspring'].mean():.2f}")
print(f"Note: organisms still alive at end of simulation are not included.")

<a id="limitations"></a>
## 7. Limitations

- **Correlation, not causation**: Observed correlations between energy and outcomes do not establish causal relationships. Known confounds include reverse causation (reproduction reduces entropy via POKE, so low entropy in reproducers may be a consequence rather than a cause) and survivorship bias (organisms that survive long enough to accumulate energy are the same ones that reach the reproduction threshold).

- **Sampling resolution**: Per-organism data is sampled at intervals. Organisms that live less than one sampling interval appear only in their death snapshot, limiting observable lifetime behavior.

- **Single run**: All observations are from a single simulation run with a single configuration. Results may not generalize to other configurations, grid sizes, or mutation regimes.

- **Informational influence**: The fourth adaptation axis proposed by Zeid — informational influence, measured via transfer entropy TE_{m→x} — is not computed. In Evochora, organisms have no separation between genotype and phenotype: the genome (CODE molecules) directly *is* the executing program. This makes the decomposition into "informational variable" and "physical variable" required by transfer entropy non-trivial. Possible approaches remain to be explored.

## 8. Open Questions & Next Steps

1. **Are there adaptive mutations?** Identifying organisms that perform above average within their thermodynamic group — and whose genome differs from the majority in that group — would yield candidates for genuine adaptation. Confirming whether a mutation actually changed behavior requires manual tick-by-tick inspection in the Evochora visualizer.

2. **How does configuration affect whether adaptation emerges?** Some factors that may influence this — such as mutation rates, energy supply, and entropy limits — are configurable today. Others — such as environmental heterogeneity, natural barriers, or catastrophic events — would need to be developed first. Running the same diagnostic across different configurations and extensions would show whether specific changes produce measurable adaptation.

3. **How can informational influence be operationalized in Evochora?** The absence of a genotype/phenotype separation in Evochora is a conceptual challenge that may require novel proxies for causal influence.